In [15]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

import tensorflow as tf
from tensorflow.keras import layers, models, losses
from tensorflow.keras.callbacks import ModelCheckpoint

# [LOG] Model Versioning

## Version 1: Tiny-Baseline 
**Data:** 15/05/2026
**Fase:** Upper-Bound Baseline (Test di fattibilità hardware)

### Architettura:
- **Input:** (1, 120, 18) -> [H, W, Channels]
- **Feature Extraction:** - Conv2D (16 filtri, kernel 1x5) + MaxPooling (1x2)
    - SeparableConv2D (32 filtri, kernel 1x3) + MaxPooling (1x2)
- **Output Heads:** - `coords_head`: Dense(8) [Linear] -> X, Y per 4 persone.
    - `mask_head`: Dense(4) [Sigmoid] -> Presenza per 4 persone.

### Statistiche:
- **Parametri Totali:** ~3,500
- **Peso Modello (Float32):** 13.67 KB
- **Peso Stimato (INT8 Quantized):** ~3.5 KB
- **Performance (Epoca 15):** - `val_loss`: 3.79
    - `val_coords_loss`: 3.35 (Errore spaziale medio ~1.83m)
    - `val_mask_loss`: 0.88

----

# [GUIDA] Gerarchia del Fine-Tuning per Edge AI (ESP32-S3)

Nel TinyML non possiamo ingrandire la rete a caso, perché siamo limitati da 400KB di RAM e dalla latenza. Le modifiche seguono un ordine di priorità basato sul **Costo Hardware**.

### Livello 1: Costo Hardware ZERO (Modifiche di Addestramento)
Questi parametri non alterano il peso finale del file `.tflite`. Si provano per primi.
* **1. Epoche (`epochs`):** * *Cos'è:* Il tempo di studio. Quante volte la rete vede l'intero dataset.
    * *Quando usarlo:* Se la `val_loss` sta scendendo ma l'addestramento finisce troppo presto (Underfitting).
    * *Effetto:* Permette alla rete di continuare a correggere gli errori.
* **2. Learning Rate (`lr`):**
    * *Cos'è:* La "lunghezza del passo" durante la discesa del gradiente.
    * *Quando usarlo:* Se la Loss salta su e giù in modo impazzito (LR troppo alto) o se non scende per niente fin dall'inizio (LR troppo basso).
    * *Effetto:* Rende l'apprendimento più stabile o più aggressivo.

### Livello 2: Costo Hardware BASSO (Capacità / Larghezza)
* **3. Numero di Filtri (es. da 32 a 64):**
    * *Quando usarlo:* Se la rete è troppo "stupida" per capire le dinamiche della stanza e la Loss si blocca su valori alti (come nella nostra V1).
    * *Effetto:* Aumenta i parametri (Flash) e leggermente la RAM. Dà alla rete più "neuroni" per capire la trigonometria.

### Livello 3: Costo Hardware ALTO (Profondità / Latenza)
* **4. Aggiungere Layer (es. una terza Conv2D):**
    * *Cos'è:* Aggiungere step sequenziali al modello.
    * *Quando usarlo:* Solo se la rete larga non basta per estrarre concetti complessi.
    * *Effetto:* Aumenta drasticamente le operazioni matematiche (MACs). **Aumenta la latenza:** l'ESP32 ci metterà molto più tempo a calcolare ogni singolo frame. Usare con estrema cautela.

In [16]:
# ==============================================================================
# IL GENERATORE DI DATI (DATA ENGINE)
# ==============================================================================
class EEAIDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, file_paths, batch_size=2, alpha=0.05, is_training=True):
        self.file_paths = file_paths
        self.batch_size = batch_size
        self.alpha = alpha
        self.is_training = is_training
        if self.is_training:
            np.random.shuffle(self.file_paths)

    def __len__(self):
        return int(np.ceil(len(self.file_paths) / float(self.batch_size)))

    def __getitem__(self, idx):
        batch_files = self.file_paths[idx * self.batch_size:(idx + 1) * self.batch_size]
        X_batch, y_coords_batch, y_mask_batch = [], [], []

        for file_path in batch_files:
            data = np.load(file_path)
            raw_iq = data['radar_cir_iq']   
            people_xy = data['people_xy']   
            people_mask = data['people_mask'] 
            T = raw_iq.shape[0]             
            
            mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2)
            mag_reshaped = mag.reshape(T, 1, 120, 18) 
            
            bg = np.copy(mag_reshaped[0])
            decluttered = np.zeros_like(mag_reshaped)
            for t in range(T):
                bg = self.alpha * mag_reshaped[t] + (1 - self.alpha) * bg
                decluttered[t] = np.abs(mag_reshaped[t] - bg)
            X_batch.append(decluttered)

            y_coords_batch.append(people_xy.reshape(T, 8))
            y_mask_batch.append(people_mask)

        X = np.concatenate(X_batch, axis=0)
        Y_coords = np.concatenate(y_coords_batch, axis=0)
        Y_mask = np.concatenate(y_mask_batch, axis=0)
        return X, {"coords_head": Y_coords, "mask_head": Y_mask}

    def on_epoch_end(self):
        if self.is_training:
            np.random.shuffle(self.file_paths)

# --- INIZIALIZZAZIONE ---
train_indices = [22, 0, 1, 2, 3, 10, 14, 16, 17, 18, 19, 21, 8, 9, 12, 5, 6, 4]
val_indices = [23, 7, 11, 13, 15, 20]

tutti_i_file = glob.glob("dataset/data/*.npz")
train_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in train_indices]
val_files = [f for f in tutti_i_file if int(os.path.basename(f).replace("window_", "").replace(".npz", "")) in val_indices]

BATCH_SIZE = 16 
train_gen = EEAIDataGenerator(train_files, batch_size=BATCH_SIZE, alpha=0.05, is_training=True)
val_gen = EEAIDataGenerator(val_files, batch_size=BATCH_SIZE, alpha=0.05, is_training=False)

print(f"Motore pronto: {len(train_files)} file di Train, {len(val_files)} file di Validation.")

Motore pronto: 18 file di Train, 6 file di Validation.


In [17]:
def masked_mse(y_true, y_pred):
    """
    Calcola l'errore sulle coordinate (MSE). 
    In futuro potremo azzerarlo se la maschera è 0.
    """
    return losses.mean_squared_error(y_true, y_pred)

In [18]:
# ==============================================================================
# ARCHITETTURA EEAI-NET V2 
# ==============================================================================
def build_eeai_model_v2(n_radars=6, n_antennas=3, n_bins=120):
    input_channels = n_radars * n_antennas
    inputs = layers.Input(shape=(1, n_bins, input_channels), name="radar_input")

    # Layer più larghi! 32 invece di 16, 64 invece di 32.
    x = layers.Conv2D(32, (1, 5), padding='same', activation='relu', name="conv_base")(inputs)
    x = layers.MaxPooling2D((1, 2), name="pool_1")(x) 

    x = layers.SeparableConv2D(64, (1, 3), padding='same', activation='relu', name="sep_conv_1")(x)
    x = layers.MaxPooling2D((1, 2), name="pool_2")(x) 

    x = layers.GlobalAveragePooling2D(name="gap")(x)

    # Collo di bottiglia allargato
    common_feat = layers.Dense(64, activation='relu', name="features")(x)

    coords_output = layers.Dense(8, activation='linear', name="coords_head")(common_feat)
    mask_output = layers.Dense(4, activation='sigmoid', name="mask_head")(common_feat)

    return models.Model(inputs=inputs, outputs=[coords_output, mask_output], name="EEAI_Net_v2")

model_v2 = build_eeai_model_v2()
print("\n--- ARCHITETTURA V2 ---")
model_v2.summary()

# Compilazione
model_v2.compile(
    optimizer='adam',
    loss={"coords_head": "mse", "mask_head": "binary_crossentropy"},
    loss_weights={"coords_head": 1.0, "mask_head": 0.5}
)

# Salviamo un file col nome V2 per non sovrascrivere la V1!
checkpoint_v2 = ModelCheckpoint("eeai_best_model_v2.keras", monitor="val_loss", save_best_only=True, verbose=1)

# FUOCO ALLE POLVERI
EPOCHS = 50 
print("\n--- INIZIO ADDESTRAMENTO---")
history_v2 = model_v2.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=[checkpoint_v2],
    verbose=1
)
print("--- ADDESTRAMENTO COMPLETATO ---")


--- ARCHITETTURA V2 ---


Model: "EEAI_Net_v2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ radar_input         │ (None, 1, 120,    │          0 │ -                 │
│ (InputLayer)        │ 18)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_base (Conv2D)  │ (None, 1, 120,    │      2,912 │ radar_input[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_1              │ (None, 1, 60, 32) │          0 │ conv_base[0][0]   │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sep_conv_1          │ (None, 1, 60, 64) │      2,208 │ pool_1[0][0]      │
│ (SeparableConv2D)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_2              │ (None, 1, 30, 64) │          0 │ sep_conv_1[0][0]  │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gap                 │ (None, 64)        │          0 │ pool_2[0][0]      │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ features (Dense)    │ (None, 64)        │      4,160 │ gap[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ coords_head (Dense) │ (None, 8)         │        520 │ features[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mask_head (Dense)   │ (None, 4)         │        260 │ features[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 10,060 (39.30 KB)

 Trainable params: 10,060 (39.30 KB)

 Non-trainable params: 0 (0.00 B)


--- INIZIO ADDESTRAMENTO---
Epoch 1/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18s/step - coords_head_loss: 25.4132 - loss: 21.9248 - mask_head_loss: 0.8288
Epoch 1: val_loss improved from None to 10.21889, saving model to eeai_best_model_v2.keras
2/2 ━━━━━━━━━━━━━━━━━━━━ 30s 27s/step - coords_head_loss: 20.3949 - loss: 13.0043 - mask_head_loss: 0.8296 - val_coords_head_loss: 9.8616 - val_loss: 10.2189 - val_mask_head_loss: 0.7146
Epoch 2/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16s/step - coords_head_loss: 12.7477 - loss: 11.1284 - mask_head_loss: 0.6304
Epoch 2: val_loss improved from 10.21889 to 6.06618, saving model to eeai_best_model_v2.keras
2/2 ━━━━━━━━━━━━━━━━━━━━ 23s 21s/step - coords_head_loss: 10.2437 - loss: 6.7068 - mask_head_loss: 0.6640 - val_coords_head_loss: 5.7347 - val_loss: 6.0662 - val_mask_head_loss: 0.6630
Epoch 3/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 838ms/step - coords_head_loss: 3.9915 - loss: 4.9131 - mask_head_loss: 0.6788
Epoch 3: val_loss improved from 6.06618 to 4.81305, saving mode

### Come leggere le Loss del nostro Multi-Head Model

1. **coords_head_loss (L'Errore di Posizione - MSE)**
Misura la distanza matematica al quadrato. Esempio: se vale `3.35`, l'errore medio in metri è la radice quadrata (circa `1.83` metri). Più scende, più le X rosse si avvicinano ai pallini verdi.

2. **mask_head_loss (L'Errore di Presenza - Binary Crossentropy)**
Misura la confusione della rete sulla presenza o meno della persona (0 o 1). Più è bassa, meno fantasmi vedremo.

3. **loss (La Loss Totale)**
Il voto complessivo: `coords_loss * 1.0 + mask_loss * 0.5`. L'ottimizzatore cerca di abbassare questo numero il più possibile.

4. **val_loss (validation loss)**
È l'errore che la rete commette sui dati che non ha mai visto (l'esame di fine modulo).


In [ ]:
# ==============================================================================
# VISUALIZZATORE 3.0 (Anti-Sfarfallio e Ground Truth Fixata)
# ==============================================================================
file_target = "dataset/data/window_000007.npz"

if not os.path.exists(file_target):
    print(f"ERRORE: Non trovo il file {file_target}")
else:
    data = np.load(file_target)
    raw_iq = data['radar_cir_iq'] 
    gt_coords = data['people_xy'] 
    gt_mask = data['people_mask'] 
    T = raw_iq.shape[0]

    print("Elaborazione filtri e previsioni in corso (V2)...")
    mag = np.sqrt(raw_iq[..., 0]**2 + raw_iq[..., 1]**2).reshape(T, 1, 120, 18)
    decluttered = np.zeros_like(mag)
    bg = np.copy(mag[0])
    alpha = 0.05
    for t in range(T):
        bg = alpha * mag[t] + (1 - alpha) * bg
        decluttered[t] = np.abs(mag[t] - bg)

    # Usa esplicitamente il modello V2 appena addestrato!
    preds = model_v2.predict(decluttered, verbose=0)
    p_coords = preds[0].reshape(T, 4, 2)
    p_mask = preds[1]
    print("Dati pronti! Inizializzazione Radar...")

    out = widgets.Output() 

    def draw_frame(frame_idx, soglia):
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(9, 11))
            ax.set_xlim(-0.5, 5.3); ax.set_ylim(-0.5, 7.7)
            ax.grid(True, linestyle=':', alpha=0.6)
            ax.set_title(f"Radar V2 | Frame: {frame_idx}/{T-1} | Window: 07", fontsize=14, fontweight='bold')

            stanza = plt.Rectangle((0, 0), 4.8, 7.2, linewidth=3, edgecolor='navy', facecolor='whitesmoke')
            ax.add_patch(stanza)

            for i in range(4):
                is_present = bool(gt_mask[frame_idx, i] > 0.5)
                if is_present:
                    rx, ry = gt_coords[frame_idx, i]
                    ax.scatter(rx, ry, c='limegreen', s=250, edgecolors='black', marker='o', label='REALE (GT)' if i==0 else "")
                    ax.text(rx, ry + 0.2, f"P{i+1}", color='darkgreen', fontweight='bold', ha='center')

                conf = float(p_mask[frame_idx, i])
                if conf >= soglia:
                    px, py = p_coords[frame_idx, i]
                    alpha_val = max(0.3, conf)
                    ax.scatter(px, py, c='red', s=200, marker='X', edgecolors='darkred', alpha=alpha_val, label='PREDETTO' if i==0 else "")
                    ax.text(px, py - 0.3, f"{conf*100:.0f}%", color='red', fontsize=10, ha='center', fontweight='bold')

            handles, labels = ax.get_legend_handles_labels()
            by_label = dict(zip(labels, handles))
            if by_label:
                ax.legend(by_label.values(), by_label.keys(), loc='upper right', frameon=True, shadow=True)

            plt.xlabel("X (Metri)"); plt.ylabel("Y (Metri)")
            plt.tight_layout(); plt.show()

    slider_frame = widgets.IntSlider(value=500, min=10, max=T-1, step=1, description='Frame:')
    slider_soglia = widgets.FloatSlider(value=0.50, min=0.1, max=0.99, step=0.05, description='Soglia:')

    def on_change(change):
        draw_frame(slider_frame.value, slider_soglia.value)

    slider_frame.observe(on_change, names='value')
    slider_soglia.observe(on_change, names='value')

    ui = widgets.VBox([slider_frame, slider_soglia, out])
    display(ui)
    draw_frame(slider_frame.value, slider_soglia.value)

Elaborazione filtri e previsioni in corso (V2)...
Dati pronti! Inizializzazione Radar...
